# AI Healthcare Assistant — Hugging Face & Prompt Engineering





## 1. Setup — Install Dependencies

In [ ]:
# Install required libraries
!pip install -q transformers accelerate huggingface_hub sentencepiece


In [ ]:
# Hugging Face login — only needed for gated models / higher rate limits
from huggingface_hub import notebook_login
# notebook_login()


In [ ]:
from warnings import simplefilter
simplefilter("ignore", category=UserWarning)

## 2. Load the Instruction-Tuned Model



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"  # swap this single line to change LLMs

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Model:", model.__class__.__name__)
print("Device:", model.device)


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Tokenizer: TokenizersBackend
Model: MistralForCausalLM
Device: cuda:0


In [ ]:
from transformers import GenerationConfig

chat_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def generate_response(prompt: str, max_new_tokens: int = 400, temperature: float = 0.3) -> str:
    # Create a GenerationConfig object with all parameters
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        temperature=max(temperature, 1e-5),
        do_sample=temperature > 0,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    outputs = chat_pipeline(
        prompt,
        generation_config=gen_config
    )
    full_text = outputs[0]["generated_text"]
    return full_text[len(prompt):].strip()

## 3. Task 1 — Interactive Medical Chatbot



In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical decision-support assistant for hospital staff. "
    "You provide medically cautious, evidence-based information. "
    "You never give a definitive diagnosis; you describe possibilities and recommend "
    "confirmatory tests or specialist consultation. Always flag emergencies clearly."
)

def ask_medical_bot(user_question: str, max_new_tokens: int = 350) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, clean_up_tokenization_spaces=False)
    return generate_response(prompt, max_new_tokens=max_new_tokens)



In [ ]:
# Simple interactive loop — run this cell and type questions, or type 'exit' to stop
def run_chatbot():
    print("MediAssist AI — type 'exit' to quit\n")
    while True:
        user_input = input("User: ")
        if user_input.strip().lower() in {"exit", "quit"}:
            print("Assistant: Session ended. Stay safe!")
            break
        response = ask_medical_bot(user_input)
        print("Assistant:\n", response, "\n")

run_chatbot()


MediAssist AI — type 'exit' to quit

User: Explain diabetes clearly and accurately
Assistant: Diabetes is a chronic metabolic disorder that affects how your body uses blood sugar (glucose). In people with diabetes, either the pancreas doesn't produce enough insulin (a hormone that regulates blood sugar levels) or the body becomes resistant to the effects of insulin. As a result, glucose builds up in the bloodstream instead of being used as fuel for energy.

There are three main types of diabetes:

1. Type 1 diabetes: This type usually develops in childhood or young adulthood. The pancreas stops producing insulin altogether, so people with type 1 diabetes must take daily insulin injections or use an insulin pump to manage their blood sugar levels.

2. Type 2 diabetes: This is the most common form of diabetes, affecting around 90% of people with diabetes. It typically develops in middle or older age, although it can occur at any age. Initially, the body may still produce some insulin, bu

## 4. Task 2 — Prompt Engineering (Role + Context + Constraints + Format)

**Original poor prompt:** `"Tell me about diabetes."`

**Problems:** no persona, no audience, no structure, no length limit → causes rambling, inconsistent, unstructured answers.


In [ ]:
diabetes_prompt = '''You are a board-certified endocrinologist writing patient-education material for a hospital's discharge-summary system.

Context: The reader is a newly diagnosed adult patient with no medical background.

Task: Explain diabetes clearly and accurately.

Constraints:
- Use only well-established medical facts; do not invent statistics or figures.
- Keep the entire answer under 300 words.
- Use plain, non-technical language.
- If uncertain about a detail, say so rather than guessing.

Output Format (strict Markdown, use these exact headings):
## Disease Overview
## Symptoms
## Risk Factors
## Prevention
## When to Consult a Doctor
'''

messages = [{"role": "user", "content": diabetes_prompt}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
diabetes_response = generate_response(prompt, max_new_tokens=500)
print(diabetes_response)


### Disease Overview
Diabetes is a chronic condition that affects how your body uses glucose, a type of sugar that comes from the foods you eat. In people without diabetes, a hormone called insulin helps move glucose from your bloodstream into your cells where it's used for energy. However, in people with diabetes, either their bodies don't make enough insulin or they can't use the insulin they produce effectively. This leads to high levels of glucose in the bloodstream, which over time can cause serious health problems.

There are two main types of diabetes: type 1 and type 2. Type 1 diabetes typically develops in childhood or young adulthood and occurs when the body stops producing insulin altogether. Type 2 diabetes usually appears later in life and happens when the body becomes resistant to insulin or doesn't produce enough to effectively manage blood sugar levels.

### Symptoms
The symptoms of diabetes can vary depending on the type and severity of the condition. Some common signs

In [ ]:
from IPython.display import Markdown, display
display(Markdown(diabetes_response))


### Disease Overview
Diabetes is a chronic condition that affects how your body uses glucose, a type of sugar that comes from the foods you eat. In people without diabetes, a hormone called insulin helps move glucose from your bloodstream into your cells where it's used for energy. However, in people with diabetes, either their bodies don't make enough insulin or they can't use the insulin they produce effectively. This leads to high levels of glucose in the bloodstream, which over time can cause serious health problems.

There are two main types of diabetes: type 1 and type 2. Type 1 diabetes typically develops in childhood or young adulthood and occurs when the body stops producing insulin altogether. Type 2 diabetes usually appears later in life and happens when the body becomes resistant to insulin or doesn't produce enough to effectively manage blood sugar levels.

### Symptoms
The symptoms of diabetes can vary depending on the type and severity of the condition. Some common signs include:

- Increased thirst and frequent urination
- Unexplained weight loss
- Blurred vision
- Fatigue and weakness
- Slow healing of cuts and wounds
- Tingling or numbness in the hands or feet

If you experience any of these symptoms, it's important to speak with your doctor as soon as possible to determine if you have diabetes.

### Risk Factors
While anyone can develop diabetes, certain factors increase the risk of developing the condition. These include:

- Being overweight or obese
- Having a family history of diabetes
- Leading an inactive lifestyle
- Eating a diet that's high in sugar and processed foods
- Smoking
- High blood pressure
- High cholesterol levels

It's important to be aware of these risks and take steps to manage them as much as possible to lower your chances of developing diabetes.

### Prevention
While there's no surefire way to prevent diabetes, there are several things you can do to lower your risk:

- Maintain a healthy weight through regular exercise and a balanced diet
- Quit smoking
- Limit your intake of sugary and processed foods
- Manage stress through activities like yoga or meditation
-

## 5. Task 3 — Few-Shot Prompting

Three worked examples are provided before the final input so the model learns the exact output pattern (Symptoms list + Severity) purely from demonstration.

In [ ]:
few_shot_clinical_prompt = '''Convert the clinical note into a structured summary using exactly this format:
Symptoms:
- <symptom 1>
- <symptom 2>
Severity:
<Mild/Moderate/Severe>

Example 1
Input: Patient has cough and fever.
Output:
Symptoms:
- Cough
- Fever
Severity:
Moderate

Example 2
Input: Patient reports mild headache and occasional dizziness, otherwise stable.
Output:
Symptoms:
- Headache
- Dizziness
Severity:
Mild

Example 3
Input: Patient presents with severe chest pain, shortness of breath, and sweating.
Output:
Symptoms:
- Chest pain
- Shortness of breath
- Sweating
Severity:
Severe

Now convert this note:
Input: Patient has a persistent dry cough, fatigue, and oxygen saturation of 91%.
Output:
'''

messages = [{"role": "user", "content": few_shot_clinical_prompt}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(generate_response(prompt, max_new_tokens=150))


Symptoms:
- Persistent dry cough
- Fatigue
Severity:
- N/A (symptoms are not quantified as mild, moderate, or severe)

Oxygen saturation:
- Oxygen saturation: 91%
Severity:
- Low (below 95%)

Note: The severity for symptoms is not explicitly stated in the input note, so it cannot be included in the structured summary. However, the oxygen saturation level is provided, and its severity can be inferred based on the standard definition of low oxygen saturation levels being below 95%.


## 6. Task 4 — Chain-of-Thought Prompting
The patient has fever, cough, and SpO₂ = 88%

In [ ]:
cot_prompt = '''You are a triage assistant. Think through the clinical picture step by step internally,
weighing each vital sign and symptom, but DO NOT show your reasoning steps in the output.

Patient data:
- Fever: present
- Cough: present
- Oxygen saturation (SpO2): 88%

After reasoning silently, output ONLY the final result in this exact format:
Urgency: <Low/Moderate/High/Emergency>
Recommendation: <one or two sentence action for hospital staff>
'''

messages = [{"role": "user", "content": cot_prompt}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(generate_response(prompt, max_new_tokens=120))


Urgency: Emergency
Recommendation: Admit patient immediately for oxygen therapy and further evaluation of respiratory distress. SpO2 levels below 90% indicate severe hypoxemia and require urgent intervention to prevent respiratory failure.


## 7. Task 5 — Structured JSON Output



In [ ]:
import json, re

json_prompt = '''You are a medical data extraction system. Read the patient description and output
ONLY a valid JSON object — no explanation, no markdown code fences, no extra text.

Schema:
{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}

Patient description: Patient has fever, persistent cough, and oxygen saturation of 88%.

JSON:
'''

messages = [{"role": "user", "content": json_prompt}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
raw_output = generate_response(prompt, max_new_tokens=200)
print("Raw model output:\n", raw_output)

# Defensive parsing: extract the first {...} block in case the model adds stray text
match = re.search(r"\{.*\}", raw_output, re.DOTALL)
if match:
    try:
        parsed = json.loads(match.group(0))
        print("\nParsed JSON:\n", json.dumps(parsed, indent=2))
    except json.JSONDecodeError as e:
        print("\nJSON parsing failed:", e)


Raw model output:
 ```json
{
  "disease": "COVID-19",
  "symptoms": ["fever", "persistent cough", "low oxygen saturation (88%)"],
  "risk_level": "high",
  "recommendation": "Hospitalization recommended due to low oxygen saturation. Isolation and close contact tracing advised."
}
```

Parsed JSON:
 {
  "disease": "COVID-19",
  "symptoms": [
    "fever",
    "persistent cough",
    "low oxygen saturation (88%)"
  ],
  "risk_level": "high",
  "recommendation": "Hospitalization recommended due to low oxygen saturation. Isolation and close contact tracing advised."
}


## 8. Task 6 — Model Comparison

| Model | Parameters | Best Use Case | Advantages | Limitations |
|---|---|---|---|---|
| **HuggingFaceH4/zephyr-7b-beta** | 7B | Instruction-following chat assistants (e.g. this healthcare bot) | Ungated/open access; strong instruction-following from DPO fine-tuning; good at structured-format compliance | Not medically fine-tuned — can still hallucinate clinical facts; 7B ceiling on complex multi-step reasoning |
| **mistralai/Mistral-7B-Instruct-v0.1** | 7B | General-purpose instruction tasks, base for further fine-tuning | Strong base performance for its size; widely benchmarked; efficient (grouped-query attention) | Gated — requires HF access approval; weaker instruction adherence than Zephyr (its own DPO-tuned descendant) out of the box |

**Takeaway for MediAssist AI:** Because both models share the Mistral-7B architecture, the pipeline built in Task 1 can switch between them (or a future medically fine-tuned model such as `BioMistral`) by changing only the `MODEL_NAME` string — directly solving the "cannot switch between LLMs" problem in the brief.

## 9. Task 7 — Prompt Optimization

**Poor response received:**
```
Maybe the patient has malaria.
Or maybe dengue.
Or maybe COVID.
I am not sure.
```

**Why this is poor:**
- No structure or confidence ranking — reads as the model guessing out loud.
- No grounding in the actual symptoms/vitals given, so it can't be checked against evidence.
- No differentiation between possibilities (no supporting rationale, no next steps).
- Ends in an unhelpful admission of uncertainty with no actionable recommendation.

**Prompt engineering techniques that would fix it:**
- **Role prompting** — anchor the model as a clinical assistant, not a casual guesser.
- **Context grounding** — feed it the actual patient symptoms/vitals instead of asking a vague open question.
- **Constraints** — require it to only state possibilities directly supported by the given symptoms, and forbid unsupported free-association.
- **Output formatting** — force a ranked list with a rationale and a required next action (test/referral), so "I am not sure" is replaced by "here is the confirmatory test."
- **Few-shot examples** of well-formatted differential diagnoses to lock in the pattern.

**Rewritten prompt:**

In [ ]:
optimized_prompt = '''You are a clinical assistant supporting differential diagnosis. Base your answer ONLY on the
symptoms provided below — do not introduce conditions unrelated to these symptoms.

Patient symptoms: high fever, joint pain, skin rash, recent travel to a dengue-endemic region.

Output format:
Possible Conditions (ranked by likelihood, most likely first):
1. <condition> — <one-sentence reason based on the symptoms above>
2. <condition> — <one-sentence reason>
3. <condition> — <one-sentence reason>

Recommended Next Step: <specific confirmatory test or specialist referral>

If the symptoms are too nonspecific to rank conditions, say so explicitly and recommend the single
most appropriate diagnostic test instead of listing guesses.
'''

messages = [{"role": "user", "content": optimized_prompt}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(generate_response(prompt, max_new_tokens=250))


Possible Conditions (ranked by likelihood):
1. Dengue fever - high fever, joint pain, and recent travel to a dengue-endemic region indicate the presence of dengue virus, which causes dengue fever.
2. Malaria - although less common in areas with dengue outbreaks, malaria can also present with fever and joint pain. However, the patient's recent travel history suggests that dengue is more likely.
3. Leptospirosis - while fever and joint pain are symptoms of leptospirosis, the presence of a skin rash makes dengue fever a more probable diagnosis.

Recommended Next Step: Confirmatory blood tests for dengue virus, including NS1 antigen, IgM, and IgG antibodies, should be performed to diagnose dengue fever accurately. If symptoms persist or worsen, a consultation with an infectious disease specialist may be necessary.


## 10. Bonus Challenge — Role-Aware Prompt Template

A single template + the same underlying model produces different tone/depth/detail depending on who is asking, by swapping only the **system** persona.

In [ ]:
ROLE_PERSONAS = {
    "doctor": (
        "You are a peer clinical assistant speaking to an attending physician. "
        "Use precise medical terminology, include differential considerations, and reference "
        "relevant clinical thresholds. Be concise and technical."
    ),
    "nurse": (
        "You are a clinical assistant speaking to a bedside nurse. "
        "Focus on monitoring parameters, warning signs to escalate, and care-plan actions. "
        "Use standard nursing/medical terminology but keep it practical and task-oriented."
    ),
    "medical_student": (
        "You are a teaching assistant speaking to a medical student. "
        "Explain the underlying reasoning and pathophysiology, define technical terms briefly, "
        "and use the moment as a teaching opportunity."
    ),
    "patient": (
        "You are a compassionate assistant speaking directly to a patient with no medical background. "
        "Avoid jargon, explain things simply and reassuringly, and always recommend they confirm "
        "anything important with their doctor."
    ),
}

def ask_role_aware_bot(role: str, user_question: str, max_new_tokens: int = 300) -> str:
    role_key = role.strip().lower().replace(" ", "_")
    if role_key not in ROLE_PERSONAS:
        raise ValueError(f"Unknown role '{role}'. Choose from: {list(ROLE_PERSONAS)}")
    messages = [
        {"role": "system", "content": ROLE_PERSONAS[role_key]},
        {"role": "user", "content": user_question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return generate_response(prompt, max_new_tokens=max_new_tokens)

question = "What does a blood oxygen level of 88% mean?"
for role in ROLE_PERSONAS:
    print(f"--- {role.upper()} ---")
    print(ask_role_aware_bot(role, question))
    print()


--- DOCTOR ---
As a peer clinical assistant, I would inform the attending physician that a blood oxygen level (SpO2) of 88% is below the normal range of 95-100%. This finding could indicate respiratory insufficiency or impaired gas exchange, potentially due to underlying lung pathologies such as pneumonia, chronic obstructive pulmonary disease (COPD), or congestive heart failure. Further evaluation may be warranted based on the patient's clinical presentation and other vital signs, including respiratory rate, heart rate, and blood pressure. In cases where SpO2 persistently remains below 90%, supplemental oxygen therapy may be necessary to prevent hypoxemia and associated complications such as organ dysfunction and tissue damage.

--- NURSE ---
As a clinical assistant, I can inform you that a blood oxygen level of 88% is below the normal range of 95-100%. This indicates that the patient's body is not getting enough oxygen, which can lead to symptoms such as shortness of breath, confusio

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


As a teaching assistant, I'd be happy to explain this to you! A blood oxygen level of 88% is lower than the normal range of 95-100%. This can indicate that there is a decreased amount of oxygen being carried in the bloodstream due to a variety of factors. One possible explanation is that the lungs are not effectively exchanging oxygen for carbon dioxide during breathing, which can occur in conditions such as chronic obstructive pulmonary disease (COPD) or pneumonia. Another possibility is that there is a decrease in red blood cell production, which can lead to anemia. Regardless of the cause, a low blood oxygen level can have serious consequences for the body, including shortness of breath, fatigue, and organ damage if left untreated. As a medical student, it's important to understand the significance of abnormal vital signs like this and to know how to properly assess and manage patients with these symptoms.

--- PATIENT ---
A blood oxygen level of 88% is lower than the normal range, 